# 02 — Article-specific AI reanalysis of primary scholarly sources

This notebook performs a focused reanalysis of the sources cited in the current article rather than a general dissertation-wide literature review. Each source is coded against the article's analytical framework and the reviewers' comments.

The analysis covers:

- five analytical approaches: archetypal, narrative, audiovisual, sociocultural, and reception-oriented;
- four model dimensions: agency, normativity, identity, and community;
- concepts and definitions relevant to the article;
- direct evidentiary support for specific claims;
- separate source functions in the Literature Review, Results, and Discussion sections;
- an audit of claims already present in the manuscript;
- cross-source comparison (for example, Deleuze ↔ Musiienko, Smith ↔ Eder, Hall ↔ Morley ↔ Staiger).

Core rule: if a source does not directly support a claim, the analysis must record that limitation rather than compensate for it with model inference.

The default repository layout uses `data/article_reanalysis` for collected sources and the manifest, `data/article` for the manuscript and reviewer comments, `outputs/article_reanalysis` for the workbook, and `.cache/article_reanalysis` for cached model responses. Every path can be overridden in the configuration cell through environment variables.


In [ ]:
from pathlib import Path
import os, re, json, time
from getpass import getpass
import openpyxl
from docx import Document
from openai import OpenAI

def find_project_root(start=None):
    current = (Path(start) if start else Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "scripts").is_dir() and (candidate / "LICENSE").exists():
            return candidate
    return current


def env_path(name, default):
    value = os.getenv(name, "").strip()
    return Path(value).expanduser().resolve() if value else Path(default).resolve()


PROJECT_ROOT = env_path("AGENCY_PROJECT_ROOT", find_project_root())
DATA_DIR = env_path("AGENCY_DATA_DIR", PROJECT_ROOT / "data")
OUTPUTS_DIR = env_path("AGENCY_OUTPUT_DIR", PROJECT_ROOT / "outputs")
CACHE_ROOT = env_path("AGENCY_CACHE_DIR", PROJECT_ROOT / ".cache")
ARTICLE_DATA_DIR = env_path("ARTICLE_DATA_DIR", DATA_DIR / "article_reanalysis")

COLLECTED_DIR = env_path("ARTICLE_SOURCES_DIR", ARTICLE_DATA_DIR / "sources")
MANIFEST_XLSX = env_path("ARTICLE_MANIFEST_XLSX", ARTICLE_DATA_DIR / "article_sources_manifest.xlsx")
ARTICLE_DOCX = env_path("ARTICLE_DOCX", DATA_DIR / "article" / "paper.docx")
REVIEWER_TXT = env_path("REVIEWER_TXT", DATA_DIR / "article" / "reviewer.txt")
OUTPUT_XLSX = env_path("ARTICLE_OUTPUT_XLSX", OUTPUTS_DIR / "article_reanalysis" / "article_reanalysis_results.xlsx")
CACHE_DIR = env_path("ARTICLE_CACHE_DIR", CACHE_ROOT / "article_reanalysis")

CHUNK_MODEL = "gpt-5.4-mini"
SOURCE_MODEL = "gpt-5.4"
SYNTHESIS_MODEL = "gpt-5.4"
CHUNK_CHARS = 45_000
CHUNK_OVERLAP = 2_500
MAX_RETRIES = 4

# One-time cache invalidation control. Set to 0 after the first successful rerun.
# When > 0, the first N sources in processing order are re-analysed from raw text,
# including all fragment-level API calls, even if old cache files exist.
FORCE_REPROCESS_FIRST_N = 0

api_key_file = os.getenv("OPENAI_API_KEY_FILE", "").strip()
API_KEY_PATH = Path(api_key_file).expanduser().resolve() if api_key_file else None

# Canonical analytical categories are defined in English for reproducibility
# and easier reuse in a public repository.
APPROACHES = {
    "archetypal": "cultural recurrence, archetypes, myth, traditional images, heroic models",
    "narrative": "character function, causality, goals, obstacles, actantial position, focalization, character arc",
    "audiovisual": "actor's body, framing, movement, editing, sound, space, star image, organization of viewer perspective",
    "sociocultural": "norms, identities, gender, class, nation, memory, trauma, power, legitimacy",
    "reception": "empirical perception, interpretation, identification, liking, moral evaluation, audience data",
}

MODEL_DIMENSIONS = {
    "agency": "decision autonomy, goal setting, initiation of action, causal influence on events, dependence on circumstances",
    "normativity": "moral evaluation of goals, means, and consequences; sanctions, rewards, and legitimation",
    "identity": "stability, multiplicity, or conflict of roles; self-definition; traumatic reconfiguration",
    "community": "relationship between individual action and the collective; loyalty, care, exclusion, distributed action, consequences for the group",
}

# Canonical concepts are English. Multilingual aliases are included only to help
# the model recognize terminology in Ukrainian, Russian, and English sources.
TARGET_CONCEPTS = {
    "film hero": ["film hero", "cinematic hero", "кіногерой", "киногерой"],
    "hero": ["hero", "герой"],
    "character": ["character", "fictional character", "персонаж"],
    "film character": ["film character", "cinematic character", "кіноперсонаж", "киноперсонаж"],
    "protagonist": ["protagonist", "протагоніст", "протагонист"],
    "antihero": ["antihero", "anti-hero", "антигерой"],
    "archetype": ["archetype", "архетип"],
    "heroism": ["heroism", "героїзм", "героизм"],
    "heroization": ["heroization", "heroisation", "героїзація", "героизация"],
    "postheroism": ["postheroism", "post-heroism", "постгероїчність", "постгероичность"],
    "agency": ["agency", "агентність", "агентность"],
    "normativity": ["normativity", "нормативність", "нормативность"],
    "identity": ["identity", "ідентичність", "идентичность"],
    "community": ["community", "спільнота", "сообщество"],
    "focalization": ["focalization", "focalisation", "фокалізація", "фокализация"],
    "alignment": ["alignment"],
    "allegiance": ["allegiance"],
    "identification": ["identification", "ідентифікація", "идентификация"],
    "empathy": ["empathy", "емпатія", "эмпатия"],
    "liking/sympathy": ["liking", "sympathy", "симпатія", "симпатия"],
    "representation": ["representation", "media representation", "screen representation", "cinematic representation", "репрезентація", "екранна репрезентація", "репрезентация", "экранная репрезентация"],
    "representation–reception relation": ["representation and reception", "representation–reception", "representation/reception", "репрезентація і рецепція", "репрезентація та рецепція", "репрезентация и рецепция"],
    "reception": ["reception", "рецепція", "рецепция"],
    "perception": ["perception", "сприйняття", "восприятие"],
    "viewer position": ["viewer position", "spectator position", "глядацька позиція", "зрительская позиция"],
    "mass consciousness": ["mass consciousness", "масова свідомість", "массовое сознание"],
    "movement-image": ["movement-image", "movement image", "образ-рух", "образ-движение"],
    "time-image": ["time-image", "time image", "образ-час", "образ-время"],
    "sociocultural representation": ["sociocultural representation", "соціокультурна репрезентація", "социокультурная репрезентация"],
    "digital traces": ["digital traces", "цифрові сліди", "цифровые следы"],
}


In [ ]:
def load_api_key() -> str:
    key = os.getenv("OPENAI_API_KEY", "").strip()
    if key:
        return key
    if API_KEY_PATH:
        p = Path(API_KEY_PATH)
        if p.exists():
            key = p.read_text(encoding="utf-8").strip()
            if key:
                return key
    key = getpass("OPENAI_API_KEY: ").strip()
    if not key:
        raise ValueError("API key is empty")
    return key

def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def read_text_file(path: Path) -> str:
    for enc in ("utf-8", "utf-8-sig", "cp1251", "latin-1"):
        try:
            return clean_text(path.read_text(encoding=enc))
        except UnicodeDecodeError:
            pass
    raise RuntimeError(f"Cannot decode {path}")

def iter_source_texts(source_dir: Path):
    for folder in sorted(p for p in source_dir.iterdir() if p.is_dir()):
        parts = []
        for f in sorted(folder.rglob("*.txt")):
            parts.append(f"\n\n===== FILE: {f.name} =====\n" + read_text_file(f))
        if parts:
            yield folder.name, "".join(parts)

def chunk_text(text: str, size=CHUNK_CHARS, overlap=CHUNK_OVERLAP):
    if len(text) <= size:
        return [text]
    out, start = [], 0
    while start < len(text):
        end = min(len(text), start + size)
        out.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap)
    return out

def article_paragraphs(docx_path: Path) -> list[str]:
    return [" ".join(p.text.split()) for p in Document(docx_path).paragraphs if p.text.strip()]

def cited_numbers(text: str) -> set[int]:
    nums = set()
    for group in re.findall(r"\[([^\]]+)\]", text):
        for part in re.split(r"[;,]", group):
            part = part.strip().replace("–", "-")
            m = re.fullmatch(r"(\d+)\s*-\s*(\d+)", part)
            if m:
                a, b = map(int, m.groups())
                nums.update(range(min(a,b), max(a,b)+1))
            elif part.isdigit():
                nums.add(int(part))
    return nums


In [ ]:
wb = openpyxl.load_workbook(MANIFEST_XLSX, read_only=True, data_only=True)
ws = wb["Manifest"]
ws.reset_dimensions()
it = ws.iter_rows(values_only=True)
headers = list(next(it))
ix = {h:i for i,h in enumerate(headers)}
manifest = [{h:r[i] for h,i in ix.items()} for r in it]
manifest_by_folder = {Path(str(r["copied_to"])).name:r for r in manifest if r.get("copied_to")}

paras = article_paragraphs(ARTICLE_DOCX)
article_context_by_ref = {}
for r in manifest:
    n = int(r["ref_no"])
    article_context_by_ref[n] = [p for p in paras if n in cited_numbers(p)]

reviewer_text = read_text_file(REVIEWER_TXT) if REVIEWER_TXT.exists() else ""
print("Manifest rows:", len(manifest))
print("Article paragraphs:", len(paras))


In [ ]:
EVIDENCE_ITEM = {
    "type":"object",
    "properties":{
        "claim":{"type":"string"},
        "evidence_quote":{"type":"string"},
        "location_hint":{"type":"string"},
        "support_strength":{"type":"string","enum":["direct","partial","contextual","not_supported"]},
        "caveat":{"type":"string"}
    },
    "required":["claim","evidence_quote","location_hint","support_strength","caveat"],
    "additionalProperties":False
}

DEFINITION_ITEM = {
    "type":"object",
    "properties":{
        "concept":{"type":"string"},
        "definition_status":{"type":"string","enum":["explicit","operational","implied"]},
        "definition_or_explanation":{"type":"string"},
        "verbatim_quote":{"type":"string"},
        "location_hint":{"type":"string"},
        "usable_for_our_definition":{"type":"boolean"}
    },
    "required":["concept","definition_status","definition_or_explanation","verbatim_quote","location_hint","usable_for_our_definition"],
    "additionalProperties":False
}

FIT_ITEM = {
    "type":"object",
    "properties":{
        "name":{"type":"string"},
        "fit_score":{"type":"integer","minimum":0,"maximum":4},
        "reason":{"type":"string"},
        "evidence_quote":{"type":"string"},
        "location_hint":{"type":"string"}
    },
    "required":["name","fit_score","reason","evidence_quote","location_hint"],
    "additionalProperties":False
}

SOURCE_SCHEMA = {
    "type":"object",
    "properties":{
        "source_scope":{"type":"string"},
        "central_argument":{"type":"string"},
        "approach_fit":{"type":"array","items":FIT_ITEM},
        "dimension_fit":{"type":"array","items":FIT_ITEM},
        "definitions":{"type":"array","items":DEFINITION_ITEM},
        "directly_supported_claims":{"type":"array","items":EVIDENCE_ITEM},
        "article_claim_audit":{"type":"array","items":EVIDENCE_ITEM},
        "literature_review_use":{"type":"string"},
        "results_use":{"type":"string"},
        "discussion_use":{"type":"string"},
        "comparison_targets":{"type":"array","items":{"type":"string"}},
        "what_not_to_claim":{"type":"array","items":{"type":"string"}},
        "primary_or_secondary_relation":{"type":"string"},
        "overall_relevance_to_article":{"type":"integer","minimum":0,"maximum":4}
    },
    "required":["source_scope","central_argument","approach_fit","dimension_fit","definitions","directly_supported_claims","article_claim_audit","literature_review_use","results_use","discussion_use","comparison_targets","what_not_to_claim","primary_or_secondary_relation","overall_relevance_to_article"],
    "additionalProperties":False
}


In [ ]:
def call_json(client, model, instructions, input_text, schema, schema_name, max_output_tokens=6000):
    """Call Responses API with strict JSON schema and recover from truncated JSON.

    If the model stops because max_output_tokens is exhausted, retry with a larger
    output budget. JSON decoding failures are also retried because an incomplete
    response usually manifests as an unterminated JSON string.
    """
    last = None
    token_budget = max_output_tokens

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = client.responses.create(
                model=model,
                instructions=instructions,
                input=input_text,
                text={"format":{"type":"json_schema","name":schema_name,"schema":schema,"strict":True}},
                max_output_tokens=token_budget,
            )

            # Responses API may return status='incomplete' when the output token
            # ceiling was reached. Do not try to parse that as complete JSON.
            status = getattr(r, "status", None)
            incomplete = getattr(r, "incomplete_details", None)
            if status == "incomplete":
                reason = getattr(incomplete, "reason", "unknown") if incomplete else "unknown"
                raise RuntimeError(f"Incomplete model response: {reason}")

            raw = (r.output_text or "").strip()
            if not raw:
                raise RuntimeError("Model returned empty output_text")
            return json.loads(raw)

        except (json.JSONDecodeError, RuntimeError) as e:
            last = e
            if attempt == MAX_RETRIES:
                raise
            # Complex/long sources may need more room for structured evidence.
            token_budget = min(max(token_budget * 2, token_budget + 2000), 20000)
            print(f"  Retry {attempt}/{MAX_RETRIES - 1}: {type(e).__name__}: {e}; max_output_tokens -> {token_budget}")
            time.sleep(2 ** attempt)

        except Exception as e:
            last = e
            if attempt == MAX_RETRIES:
                raise
            print(f"  Retry {attempt}/{MAX_RETRIES - 1}: {type(e).__name__}: {e}")
            time.sleep(2 ** attempt)

    raise last


def safe_cache_stem(source_id: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", source_id)[:120]


BASE_INSTRUCTIONS = f"""
You are coding primary scholarly sources for one specific article about the film hero as a sociocultural phenomenon.
Write all analytical fields in English. Preserve verbatim quotations in the original source language.
When a concept appears in Ukrainian, Russian, or another language, normalize the analytical label to the canonical English concept while preserving the original wording in quotations.

ARTICLE'S FIVE ANALYTICAL APPROACHES:
{json.dumps(APPROACHES, ensure_ascii=False, indent=2)}

ARTICLE'S FOUR MODEL DIMENSIONS:
{json.dumps(MODEL_DIMENSIONS, ensure_ascii=False, indent=2)}

TARGET CONCEPTS AND MULTILINGUAL ALIASES:
{json.dumps(TARGET_CONCEPTS, ensure_ascii=False, indent=2)}

STRICT RULES:
1. Analyse only the supplied source text and the supplied excerpts from the current article.
2. Never infer an empirical audience effect from textual, narrative, or formal film analysis alone.
3. Distinguish explicit source claims from our interpretation.
4. For every important claim, preserve the shortest sufficient quotation and a location hint.
5. If a target concept is not defined by the source, do not invent a definition.
6. If a source interprets another theorist, explicitly distinguish the primary theorist from the secondary interpreter.
7. Assess separately what the source can contribute to the Literature Review, Results, and Discussion sections.
8. Audit every supplied current-article claim as direct, partial, contextual, or not_supported.
9. Trace the five approaches and four dimensions to actual source evidence rather than forcing every source into every category.
10. Pay special attention to the distinction between character construction, the viewer position proposed by the text, and empirical audience reception.
11. Use canonical English analytical labels even when the source uses Ukrainian or Russian terminology.
12. Treat representation and reception as distinct but potentially connected analytical concepts. Explicitly extract any passage that explains how a character is represented, how audiences receive or interpret that representation, or how the source relates textual/screen representation to empirical reception. Do not infer reception from representation unless the source provides audience evidence.
13. For English-language sources, search and analyse the original English terminology directly; multilingual aliases are supplementary and must never replace the source's own wording.
"""


In [ ]:
client = OpenAI(api_key=load_api_key())
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_CACHE_DIR = CACHE_DIR / "chunks"
CHUNK_CACHE_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

# -----------------------------------------------------------------------------
# PRE-SCAN: show workload BEFORE the first paid API request
# -----------------------------------------------------------------------------
source_plan = []
for folder_name, source_text in iter_source_texts(COLLECTED_DIR):
    meta = manifest_by_folder.get(folder_name)
    if not meta:
        source_plan.append({
            "folder_name": folder_name,
            "source_id": None,
            "ref_no": None,
            "chars": len(source_text),
            "chunks": len(chunk_text(source_text)),
            "cached": False,
            "skip": True,
        })
        continue

    source_id = str(meta["source_id"])
    cache_file = CACHE_DIR / (safe_cache_stem(source_id) + ".json")
    source_plan.append({
        "folder_name": folder_name,
        "source_id": source_id,
        "ref_no": int(meta["ref_no"]),
        "chars": len(source_text),
        "chunks": len(chunk_text(source_text)),
        "cached": cache_file.exists(),
        "skip": False,
    })

valid_plan = [x for x in source_plan if not x["skip"]]

# Mark the first N sources for a one-time full re-analysis. This invalidates both
# source-level and fragment-level caches for those sources without deleting files.
for idx, x in enumerate(valid_plan, 1):
    x["force_reprocess"] = idx <= FORCE_REPROCESS_FIRST_N

uncached_plan = [x for x in valid_plan if (not x["cached"]) or x["force_reprocess"]]

# Count fragment calls still needed, taking the new fragment cache into account.
remaining_chunk_calls = 0
cached_chunks = 0
for x in uncached_plan:
    stem = safe_cache_stem(x["source_id"])
    for i in range(1, x["chunks"] + 1):
        cf = CHUNK_CACHE_DIR / f"{stem}__chunk_{i:04d}.json"
        if cf.exists() and not x.get("force_reprocess", False):
            cached_chunks += 1
        else:
            remaining_chunk_calls += 1

print("=" * 72)
print("ARTICLE SOURCE REANALYSIS — WORKLOAD")
print(f"Primary sources found:        {len(valid_plan)}")
print(f"Already fully cached:         {sum(x['cached'] for x in valid_plan)}")
print(f"Sources to process now:       {len(uncached_plan)}")
print(f"Forced full re-analysis:      {sum(x.get('force_reprocess', False) for x in valid_plan)}")
print(f"Fragments in uncached sources:{sum(x['chunks'] for x in uncached_plan)}")
print(f"Fragments already cached:     {cached_chunks}")
print(f"Fragment API calls remaining: {remaining_chunk_calls}")
print(f"Final source syntheses needed:{len(uncached_plan)}")
print("=" * 72)

# Optional compact per-source plan. Useful for spotting monographs/dissertations.
for pos, x in enumerate(valid_plan, 1):
    state = "FORCE REPROCESS" if x.get("force_reprocess", False) else ("CACHED" if x["cached"] else "TO PROCESS")
    print(f"[{pos:02d}/{len(valid_plan):02d}] ref {x['ref_no']:>2} | {x['chunks']:>3} fragment(s) | {x['chars']:,} chars | {state} | {x['source_id']}")

print("=" * 72)

# Shared schema for fragment-level extraction.
chunk_schema = {
    "type":"object",
    "properties":{
        "relevant_passages":{"type":"array","items":EVIDENCE_ITEM},
        "definitions":{"type":"array","items":DEFINITION_ITEM},
        "approach_observations":{"type":"array","items":{"type":"string"}},
        "dimension_observations":{"type":"array","items":{"type":"string"}},
        "limitations":{"type":"array","items":{"type":"string"}}
    },
    "required":["relevant_passages","definitions","approach_observations","dimension_observations","limitations"],
    "additionalProperties":False
}

# -----------------------------------------------------------------------------
# PROCESSING
# -----------------------------------------------------------------------------
source_counter = 0
for folder_name, source_text in iter_source_texts(COLLECTED_DIR):
    meta = manifest_by_folder.get(folder_name)
    if not meta:
        print("No manifest metadata for", folder_name)
        continue

    source_counter += 1
    ref_no = int(meta["ref_no"])
    citation = str(meta["reference"])
    article_claims = article_context_by_ref.get(ref_no, [])
    source_id = str(meta["source_id"])
    stem = safe_cache_stem(source_id)
    cache_file = CACHE_DIR / (stem + ".json")
    chunks = chunk_text(source_text)

    print(f"\nSOURCE {source_counter}/{len(valid_plan)} | ref {ref_no} | {source_id}")
    print(f"  Size: {len(source_text):,} chars | fragments: {len(chunks)}")

    force_reprocess = source_counter <= FORCE_REPROCESS_FIRST_N

    if cache_file.exists() and not force_reprocess:
        result = json.loads(cache_file.read_text(encoding="utf-8"))
        print("  -> full source cache found; API skipped")
    else:
        if force_reprocess:
            print("  -> forced full re-analysis; old source and fragment caches will be ignored")
        chunk_notes = []
        for i, ch in enumerate(chunks, 1):
            chunk_cache_file = CHUNK_CACHE_DIR / f"{stem}__chunk_{i:04d}.json"

            if chunk_cache_file.exists() and not force_reprocess:
                note = json.loads(chunk_cache_file.read_text(encoding="utf-8"))
                print(f"  Fragment {i}/{len(chunks)} -> cache")
            else:
                print(f"  Fragment {i}/{len(chunks)} -> API", flush=True)
                note = call_json(
                    client, CHUNK_MODEL, BASE_INSTRUCTIONS,
                    f"SOURCE ID: {source_id}\nCITATION: {citation}\nPART {i}/{len(chunks)}\n\nSOURCE TEXT:\n{ch}",
                    chunk_schema, "article_source_chunk", 5000
                )
                # Save each successful fragment immediately. If the notebook is
                # interrupted later, already-paid fragment analyses are preserved.
                chunk_cache_file.write_text(
                    json.dumps(note, ensure_ascii=False, indent=2),
                    encoding="utf-8"
                )
                print(f"  Fragment {i}/{len(chunks)} -> saved")

            chunk_notes.append(note)

        synthesis_input = {
            "source_id": source_id,
            "citation": citation,
            "current_article_paragraphs_citing_this_source": article_claims,
            "reviewer_comments": reviewer_text,
            "chunk_notes": chunk_notes,
        }
        print(f"  Final synthesis -> API ({len(chunk_notes)} fragment notes)", flush=True)
        result = call_json(
            client, SOURCE_MODEL, BASE_INSTRUCTIONS,
            json.dumps(synthesis_input, ensure_ascii=False),
            SOURCE_SCHEMA, "article_source_final", 10000
        )
        cache_file.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
        print("  Final synthesis -> saved")

    all_results.append({"ref_no":ref_no,"source_id":source_id,"citation":citation,"result":result})
    print(f"  SOURCE {source_counter}/{len(valid_plan)} OK")

print("\nAll source-level analyses completed:", len(all_results))


In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

wb = Workbook()
wb.remove(wb.active)

def add_sheet(name, rows):
    ws = wb.create_sheet(name)
    if not rows:
        return ws
    headers = list(rows[0].keys())
    ws.append(headers)
    for r in rows:
        ws.append([r.get(h, "") for h in headers])
    for c in ws[1]:
        c.font = Font(bold=True)
        c.fill = PatternFill("solid", fgColor="D9EAF7")
    for row in ws.iter_rows():
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    ws.freeze_panes = "A2"
    for col in ws.columns:
        letter = col[0].column_letter
        ws.column_dimensions[letter].width = min(70, max(12, max(len(str(c.value or "")) for c in list(col)[:100]) + 2))
    return ws

source_rows=[]; def_rows=[]; approach_rows=[]; dim_rows=[]; claim_rows=[]; audit_rows=[]; section_rows=[]
for item in all_results:
    r = item["result"]
    base = {"ref_no":item["ref_no"],"source_id":item["source_id"],"citation":item["citation"]}
    source_rows.append({**base,
        "source_scope":r["source_scope"],
        "central_argument":r["central_argument"],
        "overall_relevance_to_article":r["overall_relevance_to_article"],
        "primary_or_secondary_relation":r["primary_or_secondary_relation"],
        "comparison_targets":" | ".join(r["comparison_targets"]),
        "what_not_to_claim":" | ".join(r["what_not_to_claim"]),
    })
    for x in r["definitions"]: def_rows.append({**base, **x})
    for x in r["approach_fit"]: approach_rows.append({**base, **x})
    for x in r["dimension_fit"]: dim_rows.append({**base, **x})
    for x in r["directly_supported_claims"]: claim_rows.append({**base, **x})
    for x in r["article_claim_audit"]: audit_rows.append({**base, **x})
    section_rows += [
        {**base,"section":"Literature Review","recommended_use":r["literature_review_use"]},
        {**base,"section":"Results","recommended_use":r["results_use"]},
        {**base,"section":"Discussion","recommended_use":r["discussion_use"]},
    ]

add_sheet("Sources", source_rows)
add_sheet("Definitions", def_rows)
add_sheet("Approach_evidence", approach_rows)
add_sheet("Dimension_evidence", dim_rows)
add_sheet("Supported_claims", claim_rows)
add_sheet("Current_claim_audit", audit_rows)
add_sheet("Section_use", section_rows)
OUTPUT_XLSX.parent.mkdir(parents=True, exist_ok=True)
wb.save(OUTPUT_XLSX)
print("Saved", OUTPUT_XLSX)


## Cross-source synthesis

This stage converts individual source annotations into article-level argumentation: where authors converge, where they diverge, which concepts are primary and which are secondary interpretations, and what the proposed model adds to existing approaches.


In [ ]:
CROSS_SCHEMA = {
    "type":"object",
    "properties":{
        "themes":{"type":"array","items":{
            "type":"object",
            "properties":{
                "theme":{"type":"string"},
                "sources":{"type":"array","items":{"type":"string"}},
                "convergences":{"type":"string"},
                "differences_or_tensions":{"type":"string"},
                "relation_to_our_five_approaches":{"type":"string"},
                "relation_to_our_four_dimensions":{"type":"string"},
                "safe_synthesis_claim":{"type":"string"},
                "recommended_section":{"type":"string","enum":["Literature Review","Results","Discussion"]},
                "cautions":{"type":"string"}
            },
            "required":["theme","sources","convergences","differences_or_tensions","relation_to_our_five_approaches","relation_to_our_four_dimensions","safe_synthesis_claim","recommended_section","cautions"],
            "additionalProperties":False
        }}
    },
    "required":["themes"],
    "additionalProperties":False
}

compact = [{
    "source_id":x["source_id"],
    "central_argument":x["result"]["central_argument"],
    "approach_fit":x["result"]["approach_fit"],
    "dimension_fit":x["result"]["dimension_fit"],
    "definitions":x["result"]["definitions"],
    "supported_claims":x["result"]["directly_supported_claims"],
    "comparison_targets":x["result"]["comparison_targets"],
} for x in all_results]

cross = call_json(
    client, SYNTHESIS_MODEL,
    BASE_INSTRUCTIONS + "\nCompare sources. Never attribute one author's claim to another. Distinguish primary texts from secondary interpretations.",
    json.dumps(compact, ensure_ascii=False),
    CROSS_SCHEMA, "cross_source_synthesis", 10000
)

wb = openpyxl.load_workbook(OUTPUT_XLSX)
if "Cross_source_synthesis" in wb.sheetnames:
    del wb["Cross_source_synthesis"]
ws = wb.create_sheet("Cross_source_synthesis")
headers = ["theme","sources","convergences","differences_or_tensions","relation_to_our_five_approaches","relation_to_our_four_dimensions","safe_synthesis_claim","recommended_section","cautions"]
ws.append(headers)
for t in cross["themes"]:
    ws.append([" | ".join(t[h]) if h == "sources" else t[h] for h in headers])
for c in ws[1]:
    c.font = Font(bold=True)
    c.fill = PatternFill("solid", fgColor="D9EAF7")
for row in ws.iter_rows():
    for c in row:
        c.alignment = Alignment(wrap_text=True, vertical="top")
ws.freeze_panes = "A2"
for col in ws.columns:
    ws.column_dimensions[col[0].column_letter].width = 45
wb.save(OUTPUT_XLSX)
print("Cross-source synthesis added")
